# Figure panels for tracks data processing and effective $k_\mathrm{obs}$ fits

## Imports

Calcium tracks data processing functions are in the analysis/tracks.jl module. Functions that make final, detailed plots are in analysis/tracksPlots.jl. Functions to fit a one-step model for effective activation rates $k_\mathrm{obs}$ are in analysis/modelsML.jl. 

In [ ]:
using DataFrames: DataFrames as DF  # all methods available as DF.method
using Plots: Plots as pl
using Printf: @sprintf
using Measures: mm

# 
module tracksAnalysis
    include("analysis/tracks.jl")
end
using .tracksAnalysis: preptrajdata, findactivtimes_thresh, cleannames, get_csv_file_names

# import small utilities from the plotting scripts
module tracksPlots
    include("analysis/tracksPlots.jl")
end
using .tracksPlots: plot_trajs_activ_cdf,  plottimescdffit!


module modelsML
    include("analysis/modelsML.jl")
end
using .modelsML: expocdf, expo_loglhood_1dset


In [ ]:
# Default styles
pl.default(
    labelfontsize = 14,
    tickfontsize = 12,
    framestyle = :axes,  # https://docs.juliaplots.org/stable/gallery/gr/generated/gr-ref034/
    yminorgrid = false,
    xminorgrid = false,
    xgrid = false,
    ygrid = false,
    size = (450, 450),
    linewidth = 2.0,
    legendfontsize=10, 
    fontfamily="Arial"
)

# Custom line styles with shorter dashes
ddash = [8, 4]
dotdash = [4, 4, 4]

# Default analysis parameters
default_fluo_factor = 0.8

# Example trajectories and thresholding
All trajectories in light gray on one plot (all with annotated transition time perhaps), one colored with thresholding to emphasize the procedure. 

In [ ]:
# Main fitting and plotting function for one example
function plot_tracks_thresholding(filepath, plotfolder; 
        highlight=1, fluothresh=default_fluo_factor, 
        filterpms=[50, 30.0], cdfthresh=0.4, dosave=false
    )
    # highlight: which track to highlight. 
    greentrajs = preptrajdata(filepath; ch="MEAN_INTENSITY_CH2", filterpms=filterpms)  # Calcium fluorescence
    fi2 = replace(fi, "0." => "0-")
    fi2 = replace(fi2, ".csv" => "")

    activtimes, activfluos, thresh = findactivtimes_thresh(
        greentrajs; fracabove=fluothresh)
    backgnd = thresh / (1.0 + fluothresh)

    # Make plot
    shortnames, lblconcs = cleannames([fi])
    ppp = plot_trajs_activ_cdf(greentrajs, activtimes, activfluos, thresh, backgnd, 
                    highlight; conclbl=shortnames[1])
    
    display(ppp)
    if dosave
        pl.savefig(ppp, joinpath(plotfolder, "tracks_example_thresholding_$(fi2).pdf"))
        println("Saved tracks examples")
    end
end

In [ ]:
# Get a cleaned up, sorted list of data file names and paths
fnames, fpaths = get_csv_file_names(datadir)
shortnames, lblconcs = cleannames(fnames)

In [ ]:
datadir = joinpath("data", "JR064_tracks")
plotfolder = joinpath("figures", "final_tracks_panels")

dset_id = 3
chosenfile = fpaths[dset_id]
println("File analyzed: ", chosenfile)

# Filtering parameters: maximal start time, minimal traj length
minlength = 50
maxstarttime = 30.0
filter_params = [minlength, maxstarttime]
cdfthreshold = 0.75

plot_tracks_thresholding(
    chosenfile,
    plotfolder,
    chosenfile;
    highlight=15, 
    dset="JR064", 
    fluothresh=default_fluo_factor,
    filterpms=filter_params,
    dosave=true, 
    cdfthresh=cdfthresh
)

# CDF example fit and all CDFs with one-step fit

In [ ]:
# Main function to process all time series
function plots_cdf_example_and_all(datadir, filelist, plotfolder, highlight_i; 
            threshfrac=0.5, cdfthresh=0.5, dset="JR064", 
            filterpms=[50, 30.0], dosave=false)
    # Clean up file list to be sure
    duplicatetracks = ["10pM_2.csv", "10pM_z1.csv", "10pM_2_z1.csv"]
    rejects = collect(any(contains(x, a) for a in duplicatetracks) for x in filelist)
    #rejects = zeros(Bool, length(filelist))
    filelistclean = collect(filelist[i] for i in 1:1:length(rejects) if !rejects[i])

    # Analyze each track, then plot tracks with activ time annotated
    all_activtimes = []
    all_cdfparams = []
    all_cdfquad = []
    all_cdf3 = []
    for fi in filelistclean
        println(fi)
        path = joinpath(datadir, fi)
        greentrajs = preptrajdata(path; ch="MEAN_INTENSITY_CH2", filterpms=filterpms);  # Calcium fluorescence
        
        # Threshold the raw trajectories
        fi2 = replace(fi, "0." => "0-")
        fi2 = replace(fi2, ".csv" => "")
        fi2 = replace(fi2, "z2" => "")
        activtimes, activfluos, thresh = findactivtimes_thresh(
            greentrajs; fracabove=threshfrac, disp=false);
        
        # Julia equivalent of append is push!
        push!(all_activtimes, activtimes)

        # Fit a linear function through early times
        #cdfparams = fitexpocdf(activtimes)
        cdfparams = fitcdfslope(activtimes, cdfthresh)
        push!(all_cdfparams, cdfparams)

        # Compare to quadratic fit, using RMS error
        quadparams = fitcdfquad(activtimes, cdfthresh)
        push!(all_cdfquad, quadparams)

        # Also three-step fits
        threeparams = fitcdf3step(activtimes, cdfthresh)
        push!(all_cdf3, threeparams)
    end

    # Prepare file names
    shortnames, lblconcs = cleannames(allnames)

    # Sort trajectories by decreasing concentration (highest at the top)
    sortidx = reverse(sortperm(lblconcs), dims=1)
    shortnames = shortnames[sortidx]
    all_activtimes = all_activtimes[sortidx]
    all_cdfparams = all_cdfparams[sortidx]
    all_cdfquad = all_cdfquad[sortidx]
    all_cdf3 = all_cdf3[sortidx]

    ## Plotting
    # Make two separate plots, insets are hard to deal with otherwise
    pp1 = pl.plot(layout=(1, 1), size=(450, 450))
    # First plot, made last: cdf example
    # with linear vs quadratic fits, add inset
    plotinfo = [pp1, pp1, 1, 2]
    titl = shortnames[highlight_i]
    activtimes = all_activtimes[highlight_i]
    linpms = all_cdfparams[highlight_i]
    quadpms = all_cdfquad[highlight_i]
    pms3 = all_cdf3[highlight_i]

    plottimescdffit!(plotinfo, activtimes, linpms, quadpms, pms3; title=titl)
    display(pp1)
    dset_fig = replace(titl, " " => "_")
    if dosave
        pl.savefig(pp1, 
            joinpath(plotfolder, "cdf_fit_example_$(dset)_$(dset_fig).pdf"))
    end

    # Then, plot all cdfs together because we don't need insets
    # Plot all CDFs together
    pp2 = pl.plot(layout=(1, 1), size=(450, 450))

    # Don't include quadratic fits on these combined plots, too busy otherwise. 
    pp2 = plotallcdfs(pp2, all_activtimes, shortnames, all_cdfparams)

    display(pp2)
    if dosave
        pl.savefig(pp2, 
            joinpath(plotfolder, "all_cdf_activecells.pdf"))
    end
    return 0
end

In [ ]:
dset_id = 3
plots_cdf_example_and_all(datadir, allnames, plotfolder, dset_id; 
            threshfrac=default_fluo_factor, 
            cdfthresh=cdfthresh, dset="JR064", 
            filterpms=filter_params, dosave=true)

# Robustness analysis
Wrap in a main function, define plotting functions above (improved versions of the PLots-based ones in the scripts). 

In [ ]:
# Main analysis + plotting function for varying the fluorescence threshold
function robustness_vs_thresh_main(filelist, datadir, plotfolder; 
            cdfthresh=0.5, dset="JR064", fluothresh_default=0.5,
            filterpms=[50, 30.0], dosave=false)
    # There have been duplications in the tracks files, some tracks are the same data:
    # 1 pM and 10pM_z1 are the same
    # 1pM_2 and 10pM_2 are the same, and 10pM_2_z1 is nearly the same
    # Remove the apparently duplicate/mislabeled 10pM ones from the summary plot. 
    duplicatetracks = ["10pM_2.csv", "10pM_z1.csv", "10pM_2_z1.csv"]
    rejects = collect(any(contains(x, a) for a in duplicatetracks) for x in filelist)
    filelistclean = collect(filelist[i] for i in 1:1:length(rejects) if !rejects[i])

    nfluopts = 50
    
    # Save CDF fits for each file and each threshold fraction tried
    all_cdflin = zeros(Float64, length(filelistclean), nfluopts, 6)
    all_cdfquad = zeros(Float64, length(filelistclean), nfluopts, 6)
    all_cdf3 = zeros(Float64, length(filelistclean), nfluopts, 6)
    fluothreshrange = collect(LinRange(0.1, 1.5, nfluopts))

    for (i, fi) in enumerate(filelistclean)
        println(fi)
        path = joinpath(datadir, fi)
        greentrajs = preptrajdata(path; ch="MEAN_INTENSITY_CH2", filterpms=filterpms)  # Calcium fluorescence
        
        # Plot these raw trajectories
        fi2 = replace(fi, "0." => "0-")
        fi2 = replace(fi2, ".csv" => "")

        # Repeat analysis for a range of fluorescence thresholds
        for j in 1:1:nfluopts
            activtimes, activfluos, thresh = findactivtimes_thresh(
                greentrajs; fracabove=fluothreshrange[j], disp=false)
        
            # Fit an exponential on the CDF
            cdfparams = fitcdfslope(activtimes, cdfthresh)
            all_cdflin[i, j, 1:6] = cdfparams[1:6]

            # Compare to quadratic fit with RMS metric for goodness of fit
            cdfquad = fitcdfquad(activtimes, cdfthresh)
            all_cdfquad[i, j, 1:6] = cdfquad[1:6]

            # Try squeezing in three-step fits
            cdf3 = fitcdf3step(activtimes, cdfthresh)
            all_cdf3[i, j, 1:6] = cdf3[1:6]
        end
    end

    # Plot slope and goodness of linear vs quadratic fits
    # as a function of the threshold fraction
    # linfits, quadfits, allnames, fracrange, fname

    allplots = plotrobustness(all_cdflin, all_cdfquad, all_cdf3, filelistclean, 
                    fluothreshrange; fluothresh_default=fluothresh_default)
    
    # Display and save the resulting plots
    ppp, pp2, pp3 = allplots

    param_names = ["k", "t0"]
    for i in 1:1:2
        nm = param_names[i]
        fname = joinpath(plotfolder, "paramfits_$(nm)_vs_threshold.pdf")
        display(ppp[i])
        pl.savefig(ppp[i], fname)
    end

    fname = joinpath(plotfolder, "linear_vs_quad_fitRMS.pdf")
    display(pp2)
    if dosave
        pl.savefig(pp2, fname)
    end

    fname = joinpath(plotfolder, "best12_vs_3-step_fitRMS.pdf")
    display(pp3)
    if dosave
        pl.savefig(pp3, fname)
    end
    return 0
end


In [ ]:
robustness_vs_thresh_main(allnames, datadir, plotfolder; 
            cdfthresh=cdfthresh, dset="JR064",
            fluothresh_default=default_fluo_factor, dosave=true, 
            filterpms=filter_params)